# Simulation and Noise

This notebook covers density matrix simulation, noise models, and Pauli
expectation values.

In [1]:
import (
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/densitymatrix"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/pauli"
	"github.com/splch/goqu/sim/statevector"
)

In [2]:
func buildBell() *ir.Circuit {
	c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
	return c
}

## Density Matrix Simulation

The density matrix simulator represents quantum states as density operators.
A pure state has purity = 1.0.

In [3]:
%%
gonbui.DisplaySvg(draw.SVG(buildBell()))

q0 
 
 q1 
 
 H

In [4]:
%%
dm := densitymatrix.New(2)
dm.Evolve(buildBell())
fmt.Printf("Purity (noiseless): %.4f\n", dm.Purity())

Purity (noiseless): 1.0000


## Depolarizing Noise

Add depolarizing noise to see purity drop below 1.0 (mixed state).

In [5]:
%%
nm := noise.New()
nm.AddDefaultError(1, noise.Depolarizing1Q(0.01))
nm.AddDefaultError(2, noise.Depolarizing2Q(0.01))

dmNoisy := densitymatrix.New(2)
dmNoisy.WithNoise(nm)
dmNoisy.Evolve(buildBell())
fmt.Printf("Purity (noisy):     %.4f\n", dmNoisy.Purity())

Purity (noisy):     0.9711


## Amplitude Damping

Amplitude damping models energy relaxation (T1 decay). Apply it to a single
qubit prepared in |1> via an X gate.

In [6]:
%%
c, _ := builder.New("amp-damp", 1).X(0).Build()
gonbui.DisplaySvg(draw.SVG(c))

nmAD := noise.New()
nmAD.AddDefaultError(1, noise.AmplitudeDamping(0.1))

dmAD := densitymatrix.New(1)
dmAD.WithNoise(nmAD)
dmAD.Evolve(c)
fmt.Printf("Purity after amplitude damping: %.4f\n", dmAD.Purity())

Purity after amplitude damping: 0.8200


q0 
 
 X

## Pauli Expectation Values

Compute expectation values of Pauli operators against a statevector.
For a Bell state, `<ZZ> = 1.0` (perfect correlation).

In [7]:
%%
sv := statevector.New(2)
sv.Evolve(buildBell())

zz, _ := pauli.Parse("ZZ")
fmt.Printf("<ZZ> = %.4f\n", real(pauli.Expect(sv.StateVector(), zz)))

xx, _ := pauli.Parse("XX")
fmt.Printf("<XX> = %.4f\n", real(pauli.Expect(sv.StateVector(), xx)))

<ZZ> = 1.0000
<XX> = 1.0000


## Hamiltonian Expectation

Build a Hamiltonian as a `PauliSum` and compute its expectation value.

In [8]:
%%
sv2 := statevector.New(2)
sv2.Evolve(buildBell())

termZZ, _ := pauli.Parse("ZZ")
termXX, _ := pauli.Parse("XX")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{
	termZZ.Scale(0.5),
	termXX.Scale(0.5),
})
energy := pauli.ExpectSum(sv2.StateVector(), ham)
fmt.Printf("<H> = %.4f\n", real(energy))

<H> = 1.0000
